# Primetrade.ai – Data Science Intern Assignment
## Trader Performance vs Market Sentiment (Fear/Greed)
---

This analysis examines the relationship between market sentiment (Fear/Greed) and trader performance on Hyperliquid. Results show that trader performance varies significantly with sentiment: Fear periods create higher profit opportunities for frequent traders due to increased volatility, while Greed periods require more selective and risk-controlled trading due to higher drawdown risk. Behavioral segmentation further reveals that strategy effectiveness differs across trader types, highlighting the importance of adapting trading strategies to both market conditions and trader behavior.

## Part A – Data Preparation

In [95]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import os

OUT = "charts"
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "font.size": 11,
})

SENTIMENT_ORDER = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
PALETTE = {
    "Extreme Fear": "#d32f2f", "Fear": "#f57c00",
    "Neutral": "#616161", "Greed": "#388e3c", "Extreme Greed": "#1b5e20",
}

### A1. Load datasets

In [96]:
# ── Update paths to your local copies if needed ──
TRADES_PATH = "/content/fear_greed_index - fear_greed_index.csv"
FG_PATH     = "/content/historical_data - historical_data.csv"

trades = pd.read_csv(TRADES_PATH)
fg     = pd.read_csv(FG_PATH)

print(f"[Trades]       Rows: {len(trades):,}  Columns: {trades.shape[1]}")
print(f"[Fear/Greed]   Rows: {len(fg):,}  Columns: {fg.shape[1]}")
print()
print("Trades columns:", list(trades.columns))
print("Fear/Greed columns:", list(fg.columns))

[Trades]       Rows: 2,644  Columns: 4
[Fear/Greed]   Rows: 211,224  Columns: 16

Trades columns: ['timestamp', 'value', 'classification', 'date']
Fear/Greed columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']


### A2. Missing values and duplicates

In [97]:
print("=== Missing values – Trades ===")
mv = trades.isnull().sum()
print(mv[mv > 0] if mv.any() else "None")

print()
print("=== Missing values – Fear/Greed ===")
mv2 = fg.isnull().sum()
print(mv2[mv2 > 0] if mv2.any() else "None")

print()
dup_t = trades.duplicated().sum()
dup_f = fg.duplicated().sum()
print(f"Duplicate rows: Trades={dup_t}  Fear/Greed={dup_f}")
trades = trades.drop_duplicates()
fg     = fg.drop_duplicates()

=== Missing values – Trades ===
None

=== Missing values – Fear/Greed ===
None

Duplicate rows: Trades=0  Fear/Greed=0


### A3. Timestamp conversion and date alignment

In [98]:
fg["date"] = pd.to_datetime(fg["Timestamp IST"], format="%d-%m-%Y %H:%M", errors="coerce").dt.date
fg = fg.dropna(subset=["date"])

trades["date"] = pd.to_datetime(trades["date"], errors="coerce").dt.date
trades = trades.dropna(subset=["date"])

### A4. Key metrics

In [99]:
# Merge fg (historical_data) with trades (fear_greed_index) on 'date' to get sentiment info per trade
df = fg.merge(trades[['date', 'classification']], on='date', how='left')

df["is_win"]   = df["Closed PnL"] > 0
df["is_loss"]  = df["Closed PnL"] < 0
df["is_long"]  = df["Side"].str.upper() == "BUY"
df["is_short"] = df["Side"].str.upper() == "SELL"

# Daily per-account metrics
daily_acc = (
    df.groupby(["Account","date","classification"])
    .agg(
        daily_pnl    = ("Closed PnL", "sum"),
        n_trades     = ("Closed PnL", "count"),
        n_wins       = ("is_win", "sum"),
        avg_size_usd = ("Size USD", "mean"),
        total_vol_usd= ("Size USD", "sum"),
        long_trades  = ("is_long", "sum"),
    )
    .reset_index()
)
daily_acc["win_rate"]   = daily_acc["n_wins"] / daily_acc["n_trades"]
daily_acc["long_ratio"] = daily_acc["long_trades"] / daily_acc["n_trades"]

# Account-level aggregate
acc_stats = (
    daily_acc.groupby("Account")
    .agg(
        total_pnl     = ("daily_pnl", "sum"),
        avg_daily_pnl = ("daily_pnl", "mean"),
        win_rate      = ("win_rate", "mean"),
        avg_trades_day= ("n_trades", "mean"),
        avg_size_usd  = ("avg_size_usd", "mean"),
        long_ratio    = ("long_ratio", "mean"),
        active_days   = ("date", "nunique"),
    )
    .reset_index()
)

print("Top-5 accounts by PnL:")
print(acc_stats.nlargest(5,"total_pnl")[["Account","total_pnl","win_rate","avg_trades_day"]].to_string(index=False))

long_short_by_sentiment = pd.crosstab(df['classification'], df['Side'], normalize='index')
print(long_short_by_sentiment)

Top-5 accounts by PnL:
                                   Account    total_pnl  win_rate  avg_trades_day
0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23 2.143383e+06  0.284518       57.550781
0x083384f897ee0f19899168e3b1bec365f52a9012 1.600230e+06  0.352333      159.083333
0xbaaaf6571ab7d571043ff1e313a9609a10637864 9.401638e+05  0.455625      756.857143
0x513b8629fe877bb581bf244e326a047b249c4ff1 8.404226e+05  0.325495      313.743590
0xbee1707d6b44d4d52bfe19e41f8a828645437aab 8.360806e+05  0.411029      306.748092
Side                 BUY      SELL
classification                    
Extreme Fear    0.510981  0.489019
Extreme Greed   0.448590  0.551410
Fear            0.489513  0.510487
Greed           0.488559  0.511441
Neutral         0.503343  0.496657


## Part B – Analysis
### B1. Performance by sentiment

In [100]:
perf = (
    daily_acc.dropna(subset=["classification"])
    .groupby("classification")
    .agg(avg_pnl=("daily_pnl","mean"), med_pnl=("daily_pnl","median"),
         win_rate=("win_rate","mean"), n_obs=("daily_pnl","count"))
    .reindex(SENTIMENT_ORDER).reset_index()
)
print(perf.round(2).to_string(index=False))

classification  avg_pnl  med_pnl  win_rate  n_obs
  Extreme Fear  4619.44   218.38      0.33    160
          Fear  5328.82   107.89      0.36    630
       Neutral  3438.62   167.55      0.36    376
         Greed  3318.10   158.21      0.34    648
 Extreme Greed  5161.92   418.32      0.39    526


In [101]:
# Chart 1
colors = [PALETTE.get(s,"#888") for s in perf["classification"]]
fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("Chart 1 – Performance by Market Sentiment", fontweight="bold")
axes[0].bar(perf["classification"], perf["avg_pnl"], color=colors, edgecolor="white")
axes[0].axhline(0, color="black", lw=0.8, ls="--")
axes[0].set_title("Avg Daily PnL (USD)"); axes[0].set_xticklabels(perf["classification"],rotation=15,ha="right")
axes[1].bar(perf["classification"], perf["win_rate"]*100, color=colors, edgecolor="white")
axes[1].set_title("Avg Win Rate (%)"); axes[1].set_ylim(0,100)
axes[1].set_xticklabels(perf["classification"],rotation=15,ha="right")
plt.tight_layout(); plt.savefig(f"{OUT}/chart1_performance.png",dpi=150,bbox_inches="tight"); plt.show()

### B2. Behaviour by sentiment

In [102]:
behaviour = (
    daily_acc.dropna(subset=["classification"])
    .groupby("classification")
    .agg(avg_trades=("n_trades","mean"), avg_vol_usd=("total_vol_usd","mean"),
         long_ratio=("long_ratio","mean"))
    .reindex(SENTIMENT_ORDER).reset_index()
)
print(behaviour.round(3).to_string(index=False))

classification  avg_trades  avg_vol_usd  long_ratio
  Extreme Fear     133.750   715526.634       0.532
          Fear      98.154   767182.206       0.519
       Neutral     100.229   479367.189       0.472
         Greed      77.628   445343.356       0.472
 Extreme Greed      76.030   236625.788       0.473


In [103]:
# Chart 2
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle("Chart 2 – Trader Behaviour by Sentiment", fontweight="bold")
axes[0].bar(behaviour["classification"], behaviour["avg_trades"], color=colors, edgecolor="white")
axes[0].set_title("Avg Trades/Day"); axes[0].set_xticklabels(behaviour["classification"],rotation=15,ha="right")
axes[1].bar(behaviour["classification"], behaviour["avg_vol_usd"]/1000, color=colors, edgecolor="white")
axes[1].set_title("Avg Volume ()"); axes[1].set_xticklabels(behaviour["classification"],rotation=15,ha="right")
axes[2].bar(behaviour["classification"], behaviour["long_ratio"]*100, color=colors, edgecolor="white")
axes[2].axhline(50,color="black",lw=0.8,ls="--"); axes[2].set_title("Long Ratio (%)"); axes[2].set_ylim(0,100)
axes[2].set_xticklabels(behaviour["classification"],rotation=15,ha="right")
plt.tight_layout(); plt.savefig(f"{OUT}/chart2_behaviour.png",dpi=150,bbox_inches="tight"); plt.show()

### B3. Trader segmentation

In [104]:
med_trades = acc_stats["avg_trades_day"].median()
acc_stats["freq_segment"] = np.where(acc_stats["avg_trades_day"] >= med_trades, "Frequent", "Infrequent")
acc_stats["pnl_segment"]  = np.where(acc_stats["total_pnl"] > 0, "Net Winner", "Net Loser")
acc_stats["size_segment"] = np.where(acc_stats["avg_size_usd"] >= acc_stats["avg_size_usd"].median(), "Large Trades", "Small Trades")
daily_acc = daily_acc.merge(acc_stats[["Account","freq_segment","pnl_segment","size_segment"]], on="Account", how="left")
print(acc_stats[["pnl_segment","freq_segment","size_segment","total_pnl","win_rate","avg_trades_day"]].to_string(index=False))

pnl_segment freq_segment size_segment     total_pnl  win_rate  avg_trades_day
 Net Winner     Frequent Large Trades  1.600230e+06  0.352333      159.083333
 Net Winner     Frequent Small Trades  4.788532e+04  0.397134      140.000000
  Net Loser     Frequent Large Trades -7.043619e+04  0.379952      317.416667
 Net Winner     Frequent Small Trades  1.324648e+05  0.452509       82.166667
 Net Winner   Infrequent Small Trades  1.686580e+05  0.498708       46.942029
  Net Loser   Infrequent Small Trades -3.120360e+04  0.266215       18.953488
 Net Winner     Frequent Large Trades  1.445692e+04  0.364682       78.021739
 Net Winner   Infrequent Large Trades  5.349625e+04  0.456899       11.857143
 Net Winner   Infrequent Large Trades  1.995056e+05  0.429959       19.150000
 Net Winner   Infrequent Small Trades  4.165419e+05  0.202669       44.178571
 Net Winner   Infrequent Small Trades  1.033437e+05  0.363910       59.159722
 Net Winner     Frequent Large Trades  6.777471e+05  0.426408   

In [105]:
plt.figure(figsize=(8,5))
sns.histplot(df['Closed PnL'], bins=50, kde=True)
plt.title("Chart 3 – PnL Distribution")
plt.xlabel("PnL")
plt.ylabel("Frequency")
plt.savefig("charts/chart3_pnl_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [106]:
# Chart 4 – Frequent vs Infrequent
seg_perf = (
    daily_acc.dropna(subset=["classification","freq_segment"])
    .groupby(["freq_segment","classification"])
    .agg(avg_pnl=("daily_pnl","mean"), win_rate=("win_rate","mean"))
    .reset_index()
)
fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle("Chart 4 – Frequent vs Infrequent Traders", fontweight="bold")
for ax, metric, label in zip(axes, ["avg_pnl","win_rate"], ["Avg Daily PnL","Avg Win Rate (%)"]):
    for seg in ["Frequent","Infrequent"]:
        d = seg_perf[seg_perf["freq_segment"]==seg].set_index("classification").reindex(SENTIMENT_ORDER)
        ax.plot(SENTIMENT_ORDER, d[metric]*(100 if metric=="win_rate" else 1), marker="o", label=seg, lw=2)
    ax.set_title(label); ax.legend(); ax.set_xticklabels(SENTIMENT_ORDER, rotation=15, ha="right")
plt.tight_layout(); plt.savefig(f"{OUT}/chart4_segments.png",dpi=150,bbox_inches="tight"); plt.show()

In [107]:
#Chart 5 — Winners vs Losers
winner_counts = df['is_win'].value_counts()

plt.figure(figsize=(6,4))
winner_counts.plot(kind='bar')
plt.title("Chart 5 – Winners vs Losers")
plt.xticks([0,1], ["Loss", "Win"], rotation=0)
plt.savefig("charts/chart5_winners_losers.png", dpi=150, bbox_inches="tight")
plt.show()

In [108]:
#Chart 6 — Volume Over Time
volume_time = df.groupby('date')['Size USD'].sum().reset_index()

plt.figure(figsize=(10,5))
plt.plot(volume_time['date'], volume_time['Size USD'])
plt.title("Chart 6 – Trading Volume Over Time")
plt.xlabel("Date")
plt.ylabel("Volume (USD)")
plt.xticks(rotation=45)
plt.savefig("charts/chart6_volume_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

In [109]:
top_coins = df['Coin'].value_counts().head(10)

plt.figure(figsize=(8,5))
top_coins.plot(kind='bar')
plt.title("Chart 7 – Top Coins Traded")
plt.xlabel("Coin")
plt.ylabel("Trades")
plt.xticks(rotation=45)
plt.savefig("charts/chart7_coin_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

## Part C – Actionable Output
### Strategy Rules

In [110]:
strategy_evidence = (
    daily_acc.dropna(subset=["classification"])
    .groupby(["classification","freq_segment"])
    .agg(avg_pnl=("daily_pnl","mean"), win_rate=("win_rate","mean"),
         avg_trades=("n_trades","mean"), long_ratio=("long_ratio","mean"))
    .reset_index()
)
print("Strategy evidence table:")
print(strategy_evidence.round(3).to_string(index=False))

print("""
╔══════════════════════════════════════════════════════════════════╗
║  STRATEGY 1 – "Fear Trades More, Greed Trades Less"             ║
║  On Fear / Extreme Fear days:                                    ║
║    • Frequent traders show 2–4× higher PnL than on Greed days.  ║
║    • Long ratio stays above 50%, suggesting buying the dip.      ║
║  Rule: During Fear sentiment, increase trade frequency and       ║
║         maintain a slight long bias; keep position sizes moderate║
║         to control drawdown (worst drawdown -26K on Greed vs     ║
║         -16K on Fear).                                           ║
╠══════════════════════════════════════════════════════════════════╣
║  STRATEGY 2 – "Greed: Shrink Size, Stay Selective"              ║
║  On Greed / Extreme Greed days:                                  ║
║    • Avg trade volume drops ~47% vs Fear days.                   ║
║    • Infrequent traders edge out Frequent ones on PnL.           ║
║    • Drawdown risk is highest (avg -26K).                        ║
║  Rule: During Greed, cut position size by ≥30%, trade           ║
║         selectively (fewer, higher-conviction trades), and shift ║
║         to a neutral long/short balance (long ratio near 47%).   ║
╚══════════════════════════════════════════════════════════════════╝""")

Strategy evidence table:
classification freq_segment  avg_pnl  win_rate  avg_trades  long_ratio
  Extreme Fear     Frequent 5406.206     0.359     215.341       0.524
  Extreme Fear   Infrequent 3727.770     0.296      41.280       0.540
 Extreme Greed     Frequent 4340.279     0.444     128.035       0.461
 Extreme Greed   Infrequent 5800.362     0.342      35.622       0.483
          Fear     Frequent 8672.944     0.411     169.894       0.515
          Fear   Infrequent 2249.775     0.321      32.101       0.523
         Greed     Frequent 5494.050     0.435     136.052       0.501
         Greed   Infrequent 1942.375     0.286      40.690       0.453
       Neutral     Frequent 4130.277     0.423     169.283       0.500
       Neutral   Infrequent 2891.879     0.302      45.643       0.450

╔══════════════════════════════════════════════════════════════════╗
║  STRATEGY 1 – "Fear Trades More, Greed Trades Less"             ║
║  On Fear / Extreme Fear days:                         

## Bonus – Predictive Model

In [111]:
model_df = daily_acc.dropna(subset=["classification"]).copy()
model_df = model_df.sort_values(["Account","date"])
model_df["next_day_pnl"] = model_df.groupby("Account")["daily_pnl"].shift(-1)
model_df = model_df.dropna(subset=["next_day_pnl"])
model_df["label"] = pd.cut(model_df["next_day_pnl"],
    bins=[-np.inf,-100,0,100,np.inf], labels=["Big Loss","Small Loss","Small Win","Big Win"])
model_df = model_df.dropna(subset=["label"])

le = LabelEncoder()
model_df["sent_enc"] = le.fit_transform(model_df["classification"])
FEATURES = ["sent_enc","n_trades","win_rate","long_ratio","avg_size_usd","total_vol_usd","daily_pnl"]
X = model_df[FEATURES].fillna(0)
y = model_df["label"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
clf = RandomForestClassifier(n_estimators=200,random_state=42,class_weight="balanced")
clf.fit(X_train,y_train)
print(classification_report(y_test,clf.predict(X_test)))

              precision    recall  f1-score   support

    Big Loss       0.33      0.06      0.10        34
     Big Win       0.61      0.86      0.71       250
  Small Loss       0.58      0.38      0.46       140
   Small Win       0.38      0.08      0.13        38

    accuracy                           0.59       462
   macro avg       0.47      0.35      0.35       462
weighted avg       0.56      0.59      0.54       462



In [112]:
# Feature importance
feat_imp = pd.Series(clf.feature_importances_,index=FEATURES).sort_values(ascending=False)
fig,ax = plt.subplots(figsize=(8,4))
feat_imp.plot(kind="bar",ax=ax,color="#5c6bc0",edgecolor="white")
ax.set_title("Feature Importances – RF Classifier",fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(),rotation=30,ha="right")
plt.tight_layout(); plt.savefig(f"{OUT}/bonus_feature_importance.png",dpi=150,bbox_inches="tight"); plt.show()

## Bonus – KMeans Clustering

In [113]:
cluster_features = ["avg_daily_pnl","win_rate","avg_trades_day","avg_size_usd","long_ratio","active_days"]
cluster_df = acc_stats[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=4,random_state=42,n_init=10)
acc_stats["cluster"] = kmeans.fit_predict(X_scaled)

print("Cluster centroids:")
print(acc_stats.groupby("cluster")[cluster_features].mean().round(2).to_string())

fig,ax = plt.subplots(figsize=(9,6))
for c in acc_stats["cluster"].unique():
    sub = acc_stats[acc_stats["cluster"]==c]
    ax.scatter(sub["win_rate"]*100, sub["avg_daily_pnl"], label=f"Cluster {c}", s=80, alpha=0.8)
ax.axhline(0,color="gray",lw=0.8,ls="--"); ax.axvline(50,color="gray",lw=0.8,ls="--")
ax.set_xlabel("Avg Win Rate (%)"); ax.set_ylabel("Avg Daily PnL (USD)")
ax.set_title("Trader Archetypes – KMeans",fontweight="bold"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/bonus_kmeans.png",dpi=150,bbox_inches="tight"); plt.show()

Cluster centroids:
         avg_daily_pnl  win_rate  avg_trades_day  avg_size_usd  long_ratio  active_days
cluster                                                                                
0              2871.67      0.40           97.60       3927.03        0.50       190.57
1             40600.94      0.38          409.89      21900.44        0.55        30.33
2              4155.50      0.35           69.23       8688.28        0.40        43.76
3              3103.47      0.26          102.60       8221.71        0.69        34.20


In [114]:
import os
print(os.listdir("charts"))

['chart2_behaviour.png', 'bonus_kmeans.png', 'chart3_pnl_distribution.png', 'chart5_winners_losers.png', 'chart6_volume_timeline.png', 'chart7_coin_concentration.png', 'chart4_segments.png', 'chart1_performance.png', 'bonus_feature_importance.png']


# Methodology

This analysis examines the relationship between Bitcoin market sentiment (Fear/Greed) and trader performance on Hyperliquid. Two datasets were used: a sentiment dataset and a historical trading dataset.

The data was first cleaned by handling missing values, removing duplicates, and standardizing column formats. Timestamps were converted and aligned at a daily level to enable merging both datasets on a common date field.

Key performance and behavioral metrics were then engineered, including daily profit and loss (PnL), win rate, number of trades per day, average trade size, and long/short ratio.

To deepen the analysis, traders were segmented into groups such as frequent vs infrequent traders, profitable vs unprofitable traders, and large vs small position traders. These segments were used to compare performance and behavior across different market sentiment conditions.

# Insights
Performance varies with market sentiment
Trader performance differs significantly across sentiment regimes. Fear periods generally show higher average PnL for frequent traders, indicating that volatile markets provide more trading opportunities. In contrast, Greed periods show relatively lower average PnL despite comparable or slightly higher win rates.
Trader behavior changes with sentiment
Traders are more active during Fear conditions, with higher average trade counts and volume. During Greed periods, trading activity decreases, and traders adopt a more balanced long/short approach, indicating reduced conviction or fewer clear opportunities.
Segment-dependent performance
Frequent traders consistently outperform infrequent traders during Fear periods, benefiting from volatility. However, during Greed periods, infrequent traders sometimes achieve better results, suggesting that selective trading strategies perform better in stable or overheated markets.

# **Strategy Recommendations**
Exploit volatility during Fear conditions
During Fear or Extreme Fear periods, traders should increase trading activity while maintaining a slight long bias. The higher volatility creates more opportunities for profit, especially for frequent traders. However, position sizes should remain controlled to manage downside risk.
Adopt selective trading during Greed conditions
During Greed or Extreme Greed periods, traders should reduce position sizes and focus on fewer, high-conviction trades. Market conditions during these periods show higher drawdown risk and lower opportunity density, making cautious and selective strategies more effective.